# Grad-CAM (Gradient-weighted Class Activation Mapping)

# Setup

## Imports

In [ ]:
from typing import List
import random
import os

import matplotlib.pyplot as plt
import numpy as np

import cv2
from PIL import Image

from torchvision.transforms import v2 as T
import torchvision.transforms.functional as F

import torch
import torch.nn as nn
import torchvision.models as models

## Load Model

### Get Model File

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Path to model (Each group member may need to modify this)
MODEL_IN_DRIVE = "/content/drive/MyDrive/COMP-SCI 5530 - Principles of Data Science/Project/training/best_model/model_state_dict.pth"

In [ ]:
# Copy model from Google Drive
!cp "{MODEL_IN_DRIVE}" "/content/model_state_dict.pth"

In [ ]:
# Config
# Path to dataset (Each group member may need to modify this)
DATASET_RAW_FILEPATH_IN_DRIVE = "/content/drive/MyDrive/COMP-SCI 5530 - Principles of Data Science/Project/data_raw/deepfake-and-real-images.zip"

In [ ]:
# Copy data clean zip from Google Drive
!cp "{DATASET_RAW_FILEPATH_IN_DRIVE}" "/content/data.zip"

In [ ]:
# Unzip data raw zip
!unzip "/content/data.zip"

### Transforms

In [ ]:
class ResizeAndPad:
    def __init__(self, target_size, fill=0):
        self.target_h, self.target_w = target_size
        self.fill = fill

    def __call__(self, img):
        w, h = img.size  # PIL: (width, height)

        if w > self.target_w:
          img.thumbnail((self.target_w, h))
          w, h = img.size  # PIL: (width, height)
        if h > self.target_h:
          img.thumbnail((w, self.target_h))
          w, h = img.size  # PIL: (width, height)

        pad_h = max(self.target_h - h, 0)
        pad_w = max(self.target_w - w, 0)

        # Split padding evenly (center the image)
        padding = (
            pad_w // 2,                    # left
            pad_h // 2,                    # top
            pad_w - pad_w // 2,            # right
            pad_h - pad_h // 2             # bottom
        )

        return F.pad(img, padding, fill=self.fill)

In [ ]:
img_size = 150

# Base transforms for val, test, and inference
base_transforms = T.Compose([
  ResizeAndPad((img_size, img_size)),

  T.ToImage(),
  T.ToDtype(torch.float32, scale=True),

  T.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
  )
])

### Model

In [ ]:
class DeepFakeDetector(nn.Module):
  def __init__(self):
    super().__init__()

    backbone = models.resnet50(pretrained=False)

    self.features = nn.Sequential(*list(backbone.children())[:-1])

    self.classifier = nn.Sequential(
      nn.Linear(2048, 512),
      nn.ReLU(),
      nn.Dropout(0.2),
      nn.Linear(512, 1)
    )

  def forward(self, x):
    x = self.features(x)
    x = torch.flatten(x, 1)
    x = self.classifier(x)
    return x

## Setup GPU

In [ ]:
# Set device to GPU if available
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

## Load model

In [ ]:
# Set model
model = DeepFakeDetector()

# Load model weights
model.load_state_dict(torch.load('model_state_dict.pth', map_location=device))

# Move model to device
model = model.to(device)

# Set model to inference mode
model.eval()

# Helper Functions

In [ ]:
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

def get_faces(img_path: str) -> List[np.ndarray]:
    # image = cv2.imread(img_path)
    image = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    faces_bboxes = face_cascade.detectMultiScale(gray, 1.3, 5)

    face_images = []
    for (x, y, w, h) in faces_bboxes:
        cropped_face = image[y:y+h, x:x+w]
        face_images.append(cropped_face)

    return face_images

In [ ]:
def compute_gradcam(model, img_tensor):
    model.eval()

    # Choose target layer (last conv layer of ResNet50)
    target_layer = model.features[7][-1]

    activations = None
    gradients = None

    # Forward hook (store feature maps)
    def forward_hook(module, input, output):
        nonlocal activations
        activations = output

    # Backward hook (store gradients w.r.t feature maps)
    def backward_hook(module, grad_input, grad_output):
        nonlocal gradients
        gradients = grad_output[0]

    # Register hooks
    fwd_hook = target_layer.register_forward_hook(forward_hook)
    bwd_hook = target_layer.register_full_backward_hook(backward_hook)

    # Forward pass
    output = model(img_tensor)          # shape: [1, 1]
    score = output[:, 0]                # target class score

    # Backward pass
    model.zero_grad()
    score.backward()

    # Remove hooks
    fwd_hook.remove()
    bwd_hook.remove()

    # Convert to numpy
    activations = activations.detach()[0]   # [C, H, W]
    gradients = gradients.detach()[0]       # [C, H, W]

    # Compute weights (global average pooling of gradients)
    weights = torch.mean(gradients, dim=(1, 2))  # [C]

    # Weighted sum of feature maps
    cam = torch.zeros(activations.shape[1:], dtype=torch.float32)

    for i, w in enumerate(weights):
        cam += w * activations[i]

    # Apply ReLU
    cam = torch.relu(cam)

    # Normalize
    cam -= cam.min()
    cam /= (cam.max() + 1e-8)

    # Resize to image size
    cam = cam.cpu().numpy()
    cam = cv2.resize(cam, (224, 224), interpolation=cv2.INTER_LINEAR)

    return cam

In [ ]:
import numpy as np
import cv2

def overlay_heatmap(img, heatmap, alpha=0.5):
    # Import pil img
    img = np.array(img)

    # Normalize heatmap
    heatmap = np.maximum(heatmap, 0)
    heatmap = heatmap / (heatmap.max() + 1e-8)

    # Resize to image size
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))

    # Convert heatmap to BGR
    heatmap = cv2.applyColorMap(np.uint8(255 * heatmap), cv2.COLORMAP_JET)

    # FIX: convert heatmap BGR → RGB
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)

    # Ensure image is RGB
    if len(img.shape) == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

    overlay = cv2.addWeighted(heatmap, alpha, img, 1 - alpha, 0)

    return overlay

# Inference

**Model Classes**

0: Deep fake image

1: Real image

In [ ]:
# Example inference
def is_deep_fake_image_heatmap(model, img_path: str) -> dict:
  face_imgs = get_faces(img_path)

  faces = {}

  if not face_imgs:
    # print("Unable to detect any faces in image")
    return faces # Unable to tell if real or not so return that it is a deep fake

  for i, face_img in enumerate(face_imgs):
    # Preprocess image
    image_pil = Image.fromarray(face_img)
    image_tensor = base_transforms(image_pil).unsqueeze(0) # Apply transforms

    output = model(image_tensor) # Get prediction from model, output is raw logit
    prob = torch.sigmoid(output).item() # Get probability converting logit to sigmoid

    pred_class = 1 if prob > 0.5 else 0

    heatmap = compute_gradcam(model, image_tensor)

    faces[i] = {
        "img_original": Image.open(img_path),
        "img_cropped": image_pil,
        "heatmap": heatmap,
        "prob": prob,
        "pred_class": pred_class
    }

  return faces

In [ ]:
def show_image(img_path: str):
  faces = is_deep_fake_image_heatmap(model, img_path)

  for name, face in faces.items():
    # Create a 2x2 grid of subplots
    fig, axs = plt.subplots(1, 2)

    fig.suptitle(f"{os.path.basename(img_path)} \n Face {name + 1}/{len(faces)} | Real img: {face['pred_class'] == 1} | Probability of real: {face['prob']:.2f}", fontsize=16, y=0.9)

    # Access specific plots via indexing
    axs[0].imshow(face["img_original"])
    axs[0].set_title("Original Image")
    axs[0].axis('off')

    axs[1].imshow(overlay_heatmap(face["img_cropped"], face["heatmap"], alpha=0.3))
    axs[1].set_title("Heatmap on cropped image")
    axs[1].axis('off')

    plt.tight_layout()

    plt.show()

In [ ]:
random.seed(1)

real_dir = "/content/Dataset/Test/Real"
fake_dir = "/content/Dataset/Test/Fake"

selected_files = random.sample(os.listdir(real_dir), k=20)
for img in selected_files:
  img_path = os.path.join(real_dir, img)
  show_image(img_path)

selected_files = random.sample(os.listdir(fake_dir), k=20)
for img in selected_files:
  img_path = os.path.join(fake_dir, img)
  show_image(img_path)